In [11]:
import requests
from bs4 import BeautifulSoup

url = "https://traquantopassa.in/stazione"

response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

In [12]:
blocchi_bus = soup.select('div.flex.items-center.gap-x-4.mb-2')

for blocco in blocchi_bus:
    numero_linea = blocco.select_one('div.w-10.h-10').text.strip()
    try:
        destinazione = blocco.select_one('span.font-medium').text.strip()
    except:
        destinazione = "Stazione Fs"
    minuti = blocco.select_one('div.text-right').text.strip()

    info_extra_tag = blocco.select_one('span.text-xs')
    info_extra = info_extra_tag.text.strip() if info_extra_tag else ""

    print(f"Linea {numero_linea} - Destinazione: {destinazione} - Arriva tra: {minuti} - Info: {info_extra}")


Linea 19 - Destinazione: Piazza Dante "Pal. Regione" - Arriva tra: 0â² - Info: 
Linea 10 - Destinazione: Cognola "Centro Civico" - Arriva tra: 0â² - Info: alla tua fermata (1Âª fermata) â¢ in orario
Linea 8 - Destinazione: Mattarello Piazza Perini - Arriva tra: 3â² - Info: a 4 fermate da te  â¢ in ritardo di 4 min
Linea 1 - Destinazione: Sopramonte Piazza Oveno - Arriva tra: 3â² - Info: non ancora partito (1Âª fermata) â¢ in orario
Linea 3 - Destinazione: Conci "Villazzano 3" - Arriva tra: 4â² - Info: a 5 fermate da te  â¢ in ritardo di 11 min
Linea 4 - Destinazione: Bettini "Madonna Bianca" - Arriva tra: 13â² - Info: a 13 fermate da te  â¢ in orario
Linea 3 - Destinazione: Conci "Villazzano 3" - Arriva tra: 13â² - Info: 
Linea 19 - Destinazione: Piazza Dante "Pal. Regione" - Arriva tra: 13â² - Info: 
Linea A - Destinazione: Stazione Fs - Arriva tra: 16â² - Info: a 11 fermate da te  â¢ in ritardo di 8 min
Linea 1 - Destinazione: Sopramonte Piazza Oveno - Arriva tra: 16â

In [ ]:
# Function to preprocess the city speed data
def city_speed_preprocessing(city_name, which_result=2, center_point_lon=10.02455, center_point_lat=45.13421, name_file = None):
    # get the graph
    graph = ox.graph_from_place(city_name, network_type='drive', which_result= which_result)
    graph = ox.project_graph(graph)
    graph = ox.add_edge_speeds(graph)
    graph = ox.add_edge_travel_times(graph)
    gdf_nodes, gdf_edges = ox.graph_to_gdfs(graph)
    #BC
    print("Computing betweenness centrality")
    bc = nx.betweenness_centrality(graph, weight='length')
    edge_bc = nx.edge_betweenness_centrality(graph, weight='length')
    scaler = MinMaxScaler()
    # scale the values between 0 and 1
    bc = scaler.fit_transform(pd.DataFrame(bc.values()))
    edge_bc = scaler.fit_transform(pd.DataFrame(edge_bc.values()))
    print("Betweenness centrality computed")
    # add the betweenness centrality to the nodes and edges
    gdf_nodes['bc'] = bc
    gdf_edges['edge_bc'] = edge_bc
    # get traffic lights
    traffic_lights = ox.features_from_place(city_name, tags={'highway':'traffic_signals'}, which_result= which_result)
    traffic_lights = traffic_lights.to_crs(gdf_nodes.crs)
    # nearest nodes
    nearest_nodes = ox.distance.nearest_nodes(graph, traffic_lights.geometry.x, traffic_lights.geometry.y)
    gdf_nodes['traffic_lights'] = gdf_nodes.index.isin(nearest_nodes)
    gdf_edges['traffic_lights'] = gdf_edges.index.get_level_values(1).isin(gdf_nodes[gdf_nodes['traffic_lights']].index)
    center_point = shapely.geometry.Point(center_point_lon, center_point_lat)
    center_point_gdf = gpd.GeoDataFrame(geometry=[center_point], crs="EPSG:4326")
    center_point_gdf = center_point_gdf.to_crs(gdf_nodes.crs)
    # add a buffer of 300 meters
    buffer = center_point_gdf.buffer(300)
    gdf_edges_clip = gdf_edges.clip(buffer.geometry[0])
    gdf_edges['traffic_center'] = gdf_edges.index.isin(gdf_edges_clip.index)
    # add the traffic speed
    gdf_edges['traffic_speed'] = gdf_edges['speed_kph'](1-gdf_edges['edge_bc']0.7)(1-gdf_edges['traffic_lights']0.2)(1-gdf_edges['traffic_center']0.7)
    gdf_edges['traffic_time'] = gdf_edges['length'] / (gdf_edges['traffic_speed'] * 1000 / 3600)
    #save to GeoJSON
    gdf_nodes.to_file("nodes_" + name_file + ".geojson", driver='GeoJSON')
    gdf_edges.to_file("edges_" + name_file + ".geojson", driver='GeoJSON')
    return gdf_nodes, gdf_edges

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2372912702.py, line 43)

In [1]:
import osmnx as ox
import networkx as nx
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import MinMaxScaler
import shapely.geometry

def city_speed_preprocessing(city_name, which_result=2, center_point_lon=10.02455, center_point_lat=45.13421, name_file = None):
    # get the graph
    graph = ox.graph_from_place(city_name, network_type='drive', which_result= which_result)
    graph = ox.project_graph(graph)
    graph = ox.add_edge_speeds(graph)
    graph = ox.add_edge_travel_times(graph)
    gdf_nodes, gdf_edges = ox.graph_to_gdfs(graph)

    return gdf_nodes, gdf_edges

In [2]:
city_speed_preprocessing("Modena")

(                        y              x  street_count highway  ref junction  \
 osmid                                                                          
 121933750    4.942146e+06  655277.579004             3     NaN  NaN      NaN   
 507968331    4.942111e+06  655293.100042             3     NaN  NaN      NaN   
 2141448236   4.941781e+06  655094.198000             1     NaN  NaN      NaN   
 1795653454   4.942719e+06  655004.064338             3     NaN  NaN      NaN   
 121933767    4.943633e+06  654121.067819             3     NaN  NaN      NaN   
 ...                   ...            ...           ...     ...  ...      ...   
 12600583181  4.939222e+06  654588.726237             3     NaN  NaN      NaN   
 12600599926  4.937536e+06  657516.502981             3     NaN  NaN      NaN   
 12600619184  4.937570e+06  657475.364335             3     NaN  NaN      NaN   
 12600619656  4.937537e+06  657436.866831             3     NaN  NaN      NaN   
 12600619662  4.937491e+06  